In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

from jppype import vscode_theme
from tqdm import tqdm

from fundus_toolkits import FundusData
from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [3]:
DATASETS_ROOT = Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/")
DATASETS_PATH = [
    DATASETS_ROOT / folder
    for folder in ["GAVE-train", "MAPLES-DR", "Fundus-AV", "LES-AV", "INSPIRE", "AV_DRIVE/test", "AV_DRIVE/training"]
]

In [4]:
from automorph_wrapper import automorph_segment_av

from fundus_vessels_toolkit.models import segment_av

if not "SKIP":
    for dataset_path in DATASETS_PATH:
        for fundus in tqdm(
            FundusData.from_folders(image=dataset_path / "1-images"), desc=f"Processing {dataset_path.name}"
        ):
            fvt_out = dataset_path / "2-av-pred_FVT" / (fundus.name + ".png")
            automorph_out = dataset_path / "2-av-pred_Automorph" / (fundus.name + ".png")

            if fvt_out.exists() and automorph_out.exists():
                continue

            fundus_cropped, roi = fundus.crop_to_roi(return_roi=True)

            if not fvt_out.exists():
                segment_av(fundus_cropped)
                fundus.update(av=fundus_cropped.av, roi=roi).write_image(av=fvt_out)

            if not automorph_out.exists():
                automorph_segment_av(fundus_cropped)
                fundus.update(av=fundus_cropped.av, roi=roi).write_image(av=automorph_out)

In [5]:
RAW = [path / "1-images" for path in DATASETS_PATH]
TOPO = [path / "3-topo" for path in DATASETS_PATH]
AV = [
    {
        "fvt": path / "2-av-pred_FVT",
        "automorph": path / "2-av-pred_Automorph",
        "gt": path / "2-av",
    }
    for path in DATASETS_PATH
]

dataset = BranchDigraphDataset.load_from_dirs(RAW, TOPO, AV, resize_to=1024, output_dir="tmp/ALL_DATA")

Found 332 branch digraphs...


In [9]:
m, sample, sample_data = dataset.jppype_show(200)

m

AttributeError: The macula segmentation was not provided.

In [7]:
digraph = sample_data.to_branch_digraph(graph=True)


In [8]:
digraph.line_list.shape

(12714, 4)